# A_03 | Coorte final e dataset supervisionado

Fluxo único e oficial: **auditar → gerar candidata → publicar → gerar artefato (partição 70/15/15 em blocos)**.

Substitui o antigo `A_03_coorte_protocolo` (mover para `notebooks/A/historico/`).
Não usa `src.protocolo.verificacoes`. Contrato: `configs/final_cohort_v1.yaml`. Partição: `configs/supervised.yaml` (`temporal`).

Partição (anos 2019, 2023–2025): `train_ref` 0,70 · `val_tune`/`val_cal`/`val_meas` 0,05 cada · `test_ref` 0,15. Anos 2020–2022 → `test_pandemic`; 2026 → `exploracao`. Testes seguem fechados até o G9.

In [ ]:
import json, shutil, sys, os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "supervised.yaml").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]  # evita 'src' antigo em cache do kernel

import pandas as pd
from src.data.final_cohort_cli import main as cohort_cli
from src.data.supervised_artifact import gerar_artefato_supervisionado

CAND = ROOT / "outputs/coorte_candidata"
PUB = ROOT / "data/analytical/coorte_final.parquet"

# Interruptores: rode tudo na 1ª vez; depois desligue o que já está pronto.
GERAR_CANDIDATA = True
PUBLICAR = True
GERAR_ARTEFATO = True

# Contagens de 24/10 (referência, não é regra): divergência pode vir de regras posteriores da coorte.
REF_24_10 = {"train_ref": 556_462, "val_tune": 39_748, "val_cal": 39_746,
             "val_meas": 39_749, "test_ref": 119_245, "test_pandemic": 2_915_829}
print("Raiz:", ROOT)

## 1. Auditoria (não remove nada)

In [ ]:
if GERAR_CANDIDATA:
    assert cohort_cli(["--mode", "audit", "--overwrite"]) == 0
    display(pd.read_csv(CAND / "resumo_regras.csv"))

## 2. Candidata + validação
Gera `outputs/coorte_candidata/`. O waterfall precisa fechar (fonte = excluídas + candidata).

In [ ]:
if GERAR_CANDIDATA:
    assert cohort_cli(["--mode", "candidate", "--overwrite"]) == 0
val = json.loads((CAND / "validacao_publicacao.json").read_text(encoding="utf-8"))
rec = val["population_reconciliation"]
print(rec)
assert rec["closed"], "waterfall não fecha: não publique"
display(pd.read_csv(CAND / "coorte_waterfall.csv"))

## 3. Publicar
Copia a candidata para `data/analytical/coorte_final.parquet` (com backup). O CLI não tem modo publish.

In [ ]:
if PUBLICAR:
    PUB.parent.mkdir(parents=True, exist_ok=True)
    if PUB.exists():
        shutil.copy2(PUB, PUB.with_suffix(".parquet.bak"))
    shutil.copy2(CAND / "coorte_final.parquet", PUB)
    shutil.copy2(CAND / "manifesto_coorte.json", PUB.with_name("coorte_final.manifest.json"))
print("Publicada:", PUB, f"{PUB.stat().st_size/1e6:.1f} MB")

## 4. Dataset supervisionado + partição-mestre
Grava `dataset_supervisionado.parquet`, `partitions_master.parquet` e `manifesto_particao.json`. Determinístico (hash do `group_id`, seed 42). Leva alguns minutos.

In [ ]:
if GERAR_ARTEFATO:
    res = gerar_artefato_supervisionado(ROOT, cohort_kind="final", exportar_particao=True, overwrite=True)
    contagens = res["partition_counts"]
    tab = pd.DataFrame({"linhas": pd.Series(contagens), "ref_24_10": pd.Series(REF_24_10)})
    tab["igual_ref"] = tab["linhas"] == tab["ref_24_10"]
    display(tab)

## 5. Conferência da razão 70/15/15
Percentuais só sobre os blocos dos anos compartilhados (os pandêmicos e 2026 ficam de fora por desenho).

In [ ]:
if GERAR_ARTEFATO:
    comp = {k: v for k, v in contagens.items() if k in ("train_ref", "val_tune", "val_cal", "val_meas", "test_ref")}
    pct = pd.Series(comp) / sum(comp.values())
    esperado = pd.Series({"train_ref": .70, "val_tune": .05, "val_cal": .05, "val_meas": .05, "test_ref": .15})
    display(pd.DataFrame({"pct": pct.round(4), "esperado": esperado}))
    assert ((pct - esperado).abs() < 0.005).all(), "razão fora de 70/5/5/5/15 (tolerância 0,5 p.p.)"
    print("OK: 70 / 5+5+5 / 15. Testes seguem fechados até o G9.")